# SIF Sentinel — Phase 1 ML Training
**TECHNICAL PROTOTYPE VALIDATION ONLY.** The dataset is synthetic; results are not real-world SIF validation. This notebook does not access the production backend.

## 1. Environment setup
Upload/extract the complete repository into Colab, or clone it before running. Set `REPO_ROOT` to the extracted repository directory.

In [ ]:
from pathlib import Path
import os, sys
REPO_ROOT = Path('/content/SIF-SENTINEL')  # change if your extracted folder differs
if not (REPO_ROOT / 'ml_training').exists():
    candidates = [p.parent for p in Path('/content').rglob('ml_training/requirements.txt')]
    if not candidates:
        raise FileNotFoundError('Upload/extract the complete repository and set REPO_ROOT.')
    REPO_ROOT = candidates[0]
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('Repository:', REPO_ROOT)

In [ ]:
%pip install -q -r ml_training/requirements.txt

## 2. Dataset upload/loading

In [ ]:
from google.colab import files
DATASET = REPO_ROOT / 'data/raw/safety_reports.csv'
if not DATASET.exists():
    print('Select safety_reports.csv')
    uploaded = files.upload()
    if 'safety_reports.csv' not in uploaded:
        raise FileNotFoundError('safety_reports.csv was not uploaded')
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    DATASET.write_bytes(uploaded['safety_reports.csv'])
print(DATASET)

## 3. Dataset validation

In [ ]:
from ml_training.src.data_loader import load_dataset
from ml_training.src.validation import audit_dataset
frame = load_dataset(DATASET)
dataset_audit = audit_dataset(frame, DATASET)
print(frame.shape, dataset_audit['dataset_sha256'])
display(frame.head())
display(frame.isna().sum().to_frame('missing'))

## 4. Leakage audit
Structured/downstream fields are explicitly excluded. Duplicate and conservative lexical near-duplicate groups are built before splitting; text-template associations are reported.

In [ ]:
from ml_training.src.leakage_audit import run_leakage_audit
groups, leakage_report = run_leakage_audit(frame, random_seed=20260909)
print(leakage_report['severity'])
print(leakage_report['conclusion'])

## 5. Preprocessing
Primary input: `report_text` only. Features: word (1,2) and character-within-word (3,5) TF-IDF.

In [ ]:
from ml_training.src.preprocessing import build_preprocessing, normalize_texts
preview_preprocessor = build_preprocessing()
print(preview_preprocessor)

## 6. Train/validation/test split
The orchestrator creates a fixed 70/15/15 group-isolated split, stratified on SIF where possible, and writes every task's class distribution to `split_report.json`.

## 7. SIF training
Logistic regression is trained on TF-IDF text. Calibration and operating-threshold selection use validation data only.

## 8. Activity training
Document-level classifier; no token-level spans are manufactured.

## 9. Hazard training
Missing values are mapped to the explicit `NONE` class.

## 10. Barrier training
Missing values are mapped to the explicit `NONE` class.

## 11. Barrier status training
Classes are `EFFECTIVE`, `FAILED`, and `UNKNOWN`.

## 12. Barrier failure training
Missing values are mapped to the explicit `NONE` class.

## 13. LSR training
Missing values are mapped to the explicit `NONE` class. This is document classification, not NER.

## 14. Evaluation
The next cell runs all seven tasks and writes validation/test metrics, per-class results, confusion matrices, and representative errors.

## 15. Calibration
Sigmoid calibration is fitted and selected using validation Brier score only; the test set remains untouched until final evaluation.

## 16. Error analysis
SIF false positives, false negatives, ambiguous cases, highly confident cases, and synthetic-template n-gram associations are exported.

## 17. Artifact export

In [ ]:
from ml_training.src.run_training import run
ARTIFACTS = REPO_ROOT / 'ml_training/artifacts'
run(DATASET, ARTIFACTS)
print((REPO_ROOT / 'ml_training/reports/FINAL_TRAINING_REPORT.md').read_text())

In [ ]:
from ml_training.src.verify_artifacts import verify
verification = verify(ARTIFACTS)
print(verification['artifact_reload'], verification['standalone_inference'])

## 18. Artifact download
Download the versioned artifacts, reports, inference source, integration contract, pinned requirements, and configuration as one ZIP.

In [ ]:
import shutil
bundle = shutil.make_archive('/content/sif_sentinel_ml_v1', 'zip', root_dir=REPO_ROOT, base_dir='ml_training')
files.download(bundle)